# Data Reading

In [0]:
csv_path = 'abfss://dbricks-sales@pysparkpractice.dfs.core.windows.net/BigMart Sales.csv'
json_path = 'abfss://dbricks-sales@pysparkpractice.dfs.core.windows.net/drivers.json'
dbutils.fs.ls(json_path)

In [0]:
df_csv = spark.read.format('csv').option('inferSchema',True).option('header',True).load(csv_path)

In [0]:
df_json = spark.read.format('json')\
    .option('inferschema',True)\
    .option('header',True)\
    .option('multiline',False)\
    .load(json_path)

# Schema Definition

###DDL SCHEMA

In [0]:
my_ddl_schema = '''
                Item_Identifier STRING,
                Item_Weight STRING,
                Item_Fat_Content STRING,
                Item_Visibility DOUBLE,
                Item_Type STRING,
                Item_MRP DOUBLE,
                Outlet_Identifier STRING,
                Outlet_Establishment_Year INTEGER,
                Outlet_Size STRING,
                Outlet_Location_Type STRING,
                Outlet_Type STRING,
                Item_Outlet_Sales DOUBLE
                '''

In [0]:
df_csv_ddl = spark.read.format('csv')\
                   .schema(my_ddl_schema)\
                    .option('header',True)\
                   .load(csv_path)

In [0]:
df_csv_ddl.printSchema()

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
my_strct_schema = StructType([
    StructField('Item_Identifier',StringType(),True),
    StructField('Item_Weight',StringType(),True),
    StructField('Item_Fat_Content',StringType(),True),
    StructField('Item_Visibility',StringType(),True),
    StructField('Item_Type',StringType(),True),
    StructField('Item_MRP',StringType(),True),
    StructField('Outlet_Identifier',StringType(),True),
    StructField('Outlet_Establishment_Year',StringType(),True),
    StructField('Outlet_Size',StringType(),True),
    StructField('Outlet_location_TYpe',StringType(),True),
    StructField('Outlet_Type',StringType(),True),
    StructField('Item_outlet_sales',StringType(),True)
]
)

df_csv_struct = spark.read.format('csv')\
                .option('header',True)\
                .schema(my_strct_schema)\
                .load(csv_path)

# SELECT

In [0]:
# df_csv.select(col('Item_Identifier'),col('Item_Weight'),col('Item_Fat_Content')).display()

# df_csv.select(df_csv.Item_Identifier,df_csv.Item_Weight,df_csv.Item_Fat_Content).display()


#ALIAS

In [0]:
# df_csv.select(col('Item_Identifier').alias('Item_ID'),col('Item_Weight').alias('Item_Wt')).display()

# FILTER/WHERE

In [0]:
# df_csv.filter((col('Item_Fat_Content')=='Regular') & (col('Item_Type') == 'Soft Drinks') & (col('Item_Weight') < 10) & (col('Outlet_Location_Type').isin('Tier 1','Tier 2')) & (col('Outlet_Size').isNull())).display()

# withColumnRenamed

In [0]:
# df_csv.withColumnRenamed('Item_Weight', 'Item_Wt').display()

#withColmn

In [0]:
# df_csv.withColumn("new_column",lit('UAS')).display()

# df_csv.withColumn('Total_Price',col('Item_Weight')*col('Item_MRP')).display()

# def acronym(val):
#     return "LF" if val == 'Low Fat' else "Reg"

# acronym_udf = udf(acronym, StringType())

# df_csv.withColumn('Item_Fat_Content',acronym_udf(col('Item_Fat_content'))).display()

In [0]:
# df_csv.withColumn('Item_Fat_Content', regexp_replace(col('Item_Fat_Content'), 'Low Fat', 'LFT'))\
#     .withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),'Regular','REGR')).display()

# Type Casting

In [0]:
# df_csv.withColumn('Item_Weight_STR', col('Item_Weight').cast('string'))\
#     .withColumn('Item_Visibility_STR',col('Item_Visibility').cast('string')).display()

# Sort/Order By

In [0]:
# df_csv.sort(col('Item_Weight').desc(),col('Item_MRP').asc()).filter(col('Item_Weight').isNotNull()).display()

# df_csv.orderBy('Item_Weight','Item_MRP').display()

# Limit

In [0]:
df_csv.limit(4).display()

In [0]:
df_csv.drop('Item_Identifier','Item_Weight').display()

#Drop Duplicates

In [0]:
 df_csv.dropDuplicates(['Item_Type']).display()

In [0]:
df_csv.distinct().display()

In [0]:
data1 = [('1','Neeraj'),('2','Niranjan')]
data2 = [('3','Ekanta'),('4','Sami')]

schema = """
        ID string,
        NAME STRING
"""

df1 = spark.createDataFrame(data1, schema)
df2 = spark.createDataFrame(data2,schema)

df1.union(df2).withColumn('ID',col('ID').cast('integer')).display()

In [0]:
data1 = [('Neeraj','1'),('Niranjan','2')]
data2 = [('3','Ekanta'),('4','Sami')]

schema1 = """
        ID string,
        NAME STRING
"""

schema2 = """
        name string,
        ID STRING
"""

df1 = spark.createDataFrame(data1,schema1)
df2 = spark.createDataFrame(data2,schema2)

df1.unionByName(df2).display()

#STRING FUNCTIONS

In [0]:
# df_csv.select(upper(col('Item_Type'))).display()

df_csv.select(lower('Item_Type').alias('Item_Type')).display()

#DATE FUNCTIONS

In [0]:
# Current Date

# df_csv.withColumn('Current Date', current_date()).display()

# Date Add

df_date = df_csv.withColumn('Current Date',current_date())\
    .withColumn('Future Date',date_add(col('Current Date'),2))\
    .withColumn("Past Date",date_add(col('Current Date'),-2))

#DATEDIFF

In [0]:
# df_date.withColumn('Date Diff', datediff(col('Current Date'), col('Past Date'))).display()

#DATE_FORMAT

In [0]:
df_date.withColumn('Current Date', date_format('Current Date','ddMMyyyy')).display()

#HANDLING NULLS

In [0]:
# df_csv.dropna('all')    # Drop the whole record if all columns are having NULLs

# df_csv.dropna('any')    # Drop the whole record if any column is having NULL

# df_csv.dropna(subset=['Outlet_Size'])

#Filling NAs

In [0]:
# df_csv.fillna('Not Available').display() #This is for all the columns, replacing all the NULLs

# df_csv.fillna('Not Available', subset=['Outlet_Size']).display()

#SPLIT AND INDEXING

In [0]:
df_csv.withColumn('Outlet_Type', split(col('Outlet_Type'),' '))\
    .withColumn('First', col('Outlet_Type')[1]).display()

#EXPLODE

In [0]:
df_csv.withColumn('Outlet_Type', split(col('Outlet_Type'),' '))\
    .withColumn('Outlet_Type',explode('Outlet_Type')).display()

#ARRAY_CONTAINS

In [0]:
df_csv.withColumn('Outlet_Type', split(col('Outlet_Type'),' '))\
    .withColumn('Flag', array_contains(col('Outlet_Type'),'Type1')).display()

#GROUPBY

In [0]:
df_csv.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP').alias('MRP'), avg('Item_MRP')).display()

#COLLECT_LIST

In [0]:
df_csv.groupBy('Item_Type').agg(collect_list('Item_Fat_Content')).display()

#PIVOT

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
df_csv.groupBy('Item_Type').pivot('Outlet_Size').agg(sum('Item_MRP')).display()

#WHEN-OTHERWISE

In [0]:
df_csv.display()

In [0]:
df_csv.withColumn('flag', when(col('Item_Type') == 'Meat', 'Non Veg')\
                            .when(col('Item_MRP') < 100, 'Veg Inexpensive')\
                            .otherwise('Veg Expensive')).select('Item_Type','Item_MRP','flag').display()

#JOINS

In [0]:
dataj1 = [('1','neeraj','d01'),
        ('2','niranjan','d02'),
        ('3','sami','d03'),
        ('4','ekanta','d03'),
        ('5','sathappan','d05'),
        ('6','bappa','d06')
        ]
schemaj1 = """
          empid string,
          name string,
          dept_id string
          """

dataj2 = [('d01','HR'),
        ('d02','lead'),
        ('d03','manager'),
        ('d04','associate'),
        ('d05','president')
        ]

schemaj2 = """
    dept_id string,
    department string
    """

df1 = spark.createDataFrame(dataj1,schemaj1)
df2 = spark.createDataFrame(dataj2,schemaj2)

df_inner = df1.join(df2,df1.dept_id == df2.dept_id,'inner')
df_left = df1.join(df2, df1.dept_id == df2.dept_id, 'left')
df_right = df1.join(df2, df1.dept_id == df2.dept_id, 'right')
df_anti = df1.join(df2,df1.dept_id == df2.dept_id, 'anti')

df_anti.display()

#ROW_NUMBER

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *
from pyspark.sql.types import *

window_spec = Window.partitionBy('Item_Type').orderBy('llItem_MRP')

df_csv.withColumn('row_num',row_number().over(window_spec))\
    .withColumn('rank', rank().over(window_spec))\
    .withColumn('dense_rank',dense_rank().over(window_spec))\
    .select('Item_Type', 'Item_MRP', 'row_num', 'rank', 'dense_rank').display()

#CUMULATIVE SUM

In [0]:
window_grouped_before = Window.partitionBy('Item_Type').orderBy('Item_Type').rowsBetween(Window.unboundedPreceding, Window.currentRow)
window_grouped_after = Window.partitionBy('Item_Type').orderBy('Item_Type').rowsBetween(Window.currentRow, Window.unboundedFollowing)


df_csv.withColumn('Grouped_Data',sum('Item_MRP').over(window_grouped_after))\
    .select('Item_Type', 'Item_MRP', 'Grouped_Data').display()

#USER DEFINED FUNCTIONS

In [0]:
def Category(val):
    return val+10

df_csv.withColumn('UDF',Category(col('Item_MRP'))).display()

#DATA WRITING

In [0]:
df_csv.display()

In [0]:
df_csv.write.format('csv').mode('overwrite')\
.save('abfss://dbricks-sales@pysparkpractice.dfs.core.windows.net/output.csv')

#ERROR

In [0]:
df_csv.write.format('csv').mode('error')\
.save('abfss://dbricks-sales@pysparkpractice.dfs.core.windows.net/output.csv')

In [0]:
df_csv.write.format('parquet').mode('append')\
.save('abfss://dbricks-sales@pysparkpractice.dfs.core.windows.net/output_parquet')

#TABLE

In [0]:
df_csv.write.format("delta") \
    .mode("append") \
    .save("abfss://dbricks-sales@pysparkpractice.dfs.core.windows.net/my_table_data")

#MANAGED VS EXTERNAL TABLES